# Football pressure events: detection, stats, animation and annotated video clips

**Pipeline in this notebook**
1. Load the player and ball frame tables, run sanity checks (units, attack direction, video alignment).
2. Compute pressure per defender with the Andrienko et al. (2017) model (D_front = 9 m, D_back = 3 m, q = 1.75).
3. Validate it (pressure should spike just before ball turnovers).
4. Detect **pressure events** and compute per-event stats, then per-team averages.
5. Render a **2D animation** of the press (shape, pressers, pressure oval).
6. Render **annotated clips from the real video** (boxes, pressers, pressure bar, minimap).

Run the cells top to bottom. Every threshold lives in the config cell. Tune them by watching clips.

In [ ]:
# Run once (installs into THIS kernel's environment), then restart the kernel if an import fails.
%pip install -q pandas numpy pyarrow scipy matplotlib opencv-python pillow

In [ ]:
import shutil, subprocess, unicodedata, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import Rectangle, Polygon, Circle
from scipy.spatial import ConvexHull
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200, "display.max_columns", 60)

# ----------------------------------------------------------------- paths
PROJECT = Path(r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half")
CACHE = PROJECT / "new_cache"
PLAYER_PATH = CACHE / "player_frame_table_named.parquet"
BALL_PATH = CACHE / "ball_frame_table_named.parquet"
VIDEO = PROJECT / "clip.mkv"
OUT_DIR = PROJECT / "pressure_out"

# ----------------------------------------------------------------- pitch / model (Andrienko et al. 2017)
PITCH_L, PITCH_W = 105.0, 68.0
D_FRONT, D_BACK, Q = 9.0, 3.0, 1.75

# ----------------------------------------------------------------- data quality
MIN_DEF = 8              # need at least this many visible defenders in a frame, else pressure is unreliable

# ----------------------------------------------------------------- pressure-event definition (TUNE THESE)
PRESS_THR = 30           # summed pressure on the ball that counts as "pressing"  (~1 defender 4 m in front)
MIN_DUR_S = 1.0          # minimum event length
MERGE_GAP_S = 0.5        # gaps shorter than this are merged into one event
PRESSER_MIN = 10         # a single defender is a "presser" if his own pressure on the ball is >= this
CTR_WINDOW_S = 5.0       # counterpress = the pressing team had the ball less than this long before the event
OUT_WINDOW_S = 3.0       # look this long after the event for a regain

# ----------------------------------------------------------------- possession runs / turnovers (validation only)
MIN_RUN, MAX_GAP = 12, 25    # frames

# ----------------------------------------------------------------- video output
PAD_S = 2.0                  # seconds of context before / after each event
VIDEO_FRAME_OFFSET = 0       # video_frame = frame_idx + OFFSET  (change if boxes are shifted in time)
SLOWMO = 1.0                 # 0.5 = half speed
OUT_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "clips").mkdir(exist_ok=True)
(OUT_DIR / "animations").mkdir(exist_ok=True)
print("config ok")

## 1. Load tables and sanity checks

In [ ]:
players = pd.read_parquet(PLAYER_PATH)
ball = pd.read_parquet(BALL_PATH)
FPS = int(round(1 / ball.t_sec.diff().median()))
MERGE_GAP = int(MERGE_GAP_S * FPS)
PRE = FPS                                   # 1 s window used for "just before a turnover"
print(f"players {players.shape} | ball {ball.shape} | FPS = {FPS}")

# team ids / names / attack direction (attack_dir=+1 -> attacks toward x = 105)
tm = players.dropna(subset=["team"]).groupby("team").agg(name=("team_name", "first"), dir=("attack_dir", "first"))
TEAM = {int(k): v for k, v in tm["name"].items()}
att_dir = {int(k): int(v) for k, v in tm["dir"].items()}
print("teams:", TEAM, "| attack_dir:", att_dir)

# 1) is attack_dir consistent with where each goalkeeper stands?
gk = players[(players.role == "goalkeeper") & players.pitch_valid].groupby("team").pitch_x.mean()
for t, gx in gk.items():
    own_goal_x = 0.0 if att_dir[int(t)] == 1 else PITCH_L
    flag = "OK " if abs(gx - own_goal_x) < 30 else "!! CHECK attack_dir"
    print(f"  {flag} {TEAM[int(t)]}: mean GK x = {gx:.1f} (own goal at x = {own_goal_x:.0f})")

# 2) which way do pitch axes run relative to the image? (used to orient the 2D views like the video)
v = players[players.pitch_valid & players.foot_px_x.notna() & players.pitch_x.notna()]
v = v.sample(min(50000, len(v)), random_state=0)
X_FLIP = np.corrcoef(v.foot_px_x, v.pitch_x)[0, 1] < 0   # pitch_x decreases to the right in the image
Y_DOWN = np.corrcoef(v.foot_px_y, v.pitch_y)[0, 1] > 0   # pitch_y increases downward in the image
print(f"X_FLIP={X_FLIP} Y_DOWN={Y_DOWN}")

# 3) video alignment
cap = cv2.VideoCapture(str(VIDEO))
assert cap.isOpened(), f"cannot open video: {VIDEO}"
VID_FPS = cap.get(cv2.CAP_PROP_FPS) or FPS
VID_N = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
VID_W, VID_H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
print(f"video: {VID_W}x{VID_H} @ {VID_FPS:.2f} fps, {VID_N} frames | table frames: 0-{int(ball.frame_idx.max())}")
print(f"largest box coords in table: x2={players.x2.max():.0f}, y2={players.y2.max():.0f}  (should fit inside {VID_W}x{VID_H})")
if abs(VID_N - (int(ball.frame_idx.max()) + 1)) > 5:
    print("!! video length and table length differ: set VIDEO_FRAME_OFFSET or check this is the processed clip")

## 2. Pressure per defender, per frame
Pressure on the **ball** and on the **ball carrier**, summed over the defending team (can exceed 100%).

In [ ]:
def pressure_pairs(tx, ty, px, py, gx, gy, d_front=D_FRONT, d_back=D_BACK, q=Q):
    # Andrienko et al. 2017: Pr = (1 - d/L)^q * 100, L = D_back + (D_front - D_back)(z^3 + 0.3z)/1.3.
    # tx,ty = target (ball/carrier), px,py = presser, gx,gy = goal the attackers are heading to.
    # z = (1 + cos(theta))/2 with theta = angle between (target->goal) and (target->presser), so that a
    # presser in front (theta=0) gets the full reach D_front (the paper's printed (1 - cos) sign is inverted).
    vx, vy = px - tx, py - ty
    ux, uy = gx - tx, gy - ty
    d = np.hypot(vx, vy)
    un = np.hypot(ux, uy)
    with np.errstate(invalid="ignore", divide="ignore"):
        cos_t = np.where((d > 0) & (un > 0), (vx * ux + vy * uy) / np.maximum(d * un, 1e-9), 1.0)
    z = (1 + cos_t) / 2
    L = d_back + (d_front - d_back) * (z ** 3 + 0.3 * z) / 1.3
    return np.clip(1 - d / L, 0, None) ** q * 100

# ---- usable ball frames: in play, valid homography, possession known, ball located on the pitch
b = ball[["frame_idx", "ball_px_x", "ball_px_y", "ball_pitch_x", "ball_pitch_y",
          "possession_team", "is_stoppage", "homography_valid"]].copy()
b = b[(~b.is_stoppage) & b.homography_valid & b.possession_team.notna() & b.ball_pitch_x.notna()].copy()
b["possession_team"] = b.possession_team.astype(int)
b["gx"] = np.where(b.possession_team.map(att_dir) == 1, PITCH_L, 0.0)   # goal being attacked
b["gy"] = PITCH_W / 2

# ---- defenders = outfield players + GK of the team NOT in possession
defs = players.loc[players.role.isin(["player", "goalkeeper"]) & players.team.notna()
                   & players.pitch_valid & players.pitch_x.notna(),
                   ["frame_idx", "track_id", "team", "pitch_x", "pitch_y"]].copy()
defs["team"] = defs.team.astype(int)
dp = defs.merge(b, on="frame_idx")
dp = dp[dp.team != dp.possession_team].copy()

car = (players.loc[players.is_carrier & players.pitch_valid, ["frame_idx", "pitch_x", "pitch_y"]]
       .rename(columns={"pitch_x": "cx", "pitch_y": "cy"}).drop_duplicates("frame_idx"))
dp = dp.merge(car, on="frame_idx", how="left")

f = lambda c: dp[c].to_numpy(float)
dp["p_ball"] = pressure_pairs(f("ball_pitch_x"), f("ball_pitch_y"), f("pitch_x"), f("pitch_y"), f("gx"), f("gy"))
dp["p_carrier"] = pressure_pairs(f("cx"), f("cy"), f("pitch_x"), f("pitch_y"), f("gx"), f("gy"))
dp["is_presser"] = dp.p_ball >= PRESSER_MIN

# ---- one row per frame
g = dp.groupby("frame_idx")
fr = pd.DataFrame({"p_ball": g.p_ball.sum(), "p_carrier": g.p_carrier.sum(min_count=1),
                   "n_def": g.size(), "n_pressers": g.is_presser.sum()})
fr = fr.join(b.set_index("frame_idx")[["possession_team", "ball_pitch_x", "ball_pitch_y", "ball_px_x", "ball_px_y"]])
fr = fr[fr.n_def >= MIN_DEF].sort_index()
fr["x_att"] = np.where(fr.possession_team.map(att_dir) == 1, fr.ball_pitch_x, PITCH_L - fr.ball_pitch_x)

print(f"frames with pressure: {len(fr)} of {len(ball)}  ({100 * len(fr) / len(ball):.0f}%)")
print(fr[["p_ball", "p_carrier", "n_def", "n_pressers"]].describe().round(1))
print("\nmean pressure by possessing team:\n", fr.groupby("possession_team")[["p_ball", "p_carrier"]].mean().rename(index=TEAM).round(1))

## 3. Validation: does pressure spike just before the ball is lost?
Indirect check used in the paper. Pressure on the ball should be clearly higher in the 1 s before a turnover than on average.

In [ ]:
bb = ball[["frame_idx", "possession_team", "is_stoppage"]].sort_values("frame_idx")
vv = bb[(~bb.is_stoppage) & bb.possession_team.notna()].copy()
vv["poss"] = vv.possession_team.astype(int)
vv["run"] = ((vv.poss != vv.poss.shift()) | (vv.frame_idx.diff() > 1)).cumsum()
runs = (vv.groupby("run").agg(team=("poss", "first"), start=("frame_idx", "min"),
                              end=("frame_idx", "max"), n=("frame_idx", "size"))
          .query("n >= @MIN_RUN").reset_index(drop=True))

turn = [(runs.team[i], runs.end[i]) for i in range(len(runs) - 1)
        if runs.team[i] != runs.team[i + 1] and 0 <= runs.start[i + 1] - runs.end[i] <= MAX_GAP]
turn = pd.DataFrame(turn, columns=["lost_by", "last_frame"])
LOST_FRAMES = set(turn.last_frame.astype(int))

pre = np.array([fr.p_ball.reindex(range(int(e) - PRE + 1, int(e) + 1)).mean() for e in turn.last_frame])
pre = pre[~np.isnan(pre)]
base = fr.p_ball.mean()
print(f"turnovers: {len(turn)} | with pressure data: {len(pre)}")
print(f"mean pressure on ball in last 1 s before turnover: {pre.mean():.1f} vs overall {base:.1f}")
if len(pre) > 10:
    print(f"share of turnovers above the overall mean: {(pre > base).mean():.0%} | "
          f"Wilcoxon signed-rank (event level): p = {wilcoxon(pre - base, alternative='greater').pvalue:.2e}")

## 4. Detect pressure events

In [ ]:
act = fr[fr.p_ball >= PRESS_THR].reset_index()
grp = ((act.frame_idx.diff() > MERGE_GAP) | (act.possession_team != act.possession_team.shift())).cumsum()
ev_p = (act.groupby(grp).agg(start=("frame_idx", "min"), end=("frame_idx", "max"),
                             poss=("possession_team", "first")).reset_index(drop=True))
ev_p["dur_s"] = (ev_p.end - ev_p.start + 1) / FPS
ev_p = ev_p[ev_p.dur_s >= MIN_DUR_S].reset_index(drop=True)
ev_p["poss"] = ev_p.poss.astype(int)
ev_p["press_team"] = 1 - ev_p.poss
print(f"pressure events: {len(ev_p)}")
print(ev_p.assign(team=ev_p.press_team.map(TEAM)).groupby("team").size())

## 5. Per-event stats

In [ ]:
bm = ball.set_index("frame_idx").sort_index()
poss_series = bm["possession_team"]
P = players[players.team.notna() & players.pitch_valid & players.role.isin(["player", "goalkeeper"])].copy()
P["team"] = P.team.astype(int)
P = P.set_index("frame_idx").sort_index()                    # visible players with pitch coords
M = dp.set_index("frame_idx").sort_index()                   # per-defender pressure
name_of = players.dropna(subset=["name"]).drop_duplicates("track_id").set_index("track_id")[["name", "number"]]

rows = []
for eid, e in enumerate(ev_p.itertuples()):
    f0, f1, pt, at = int(e.start), int(e.end), int(e.press_team), int(e.poss)
    span = fr.loc[f0:f1]
    fpk = int(span.p_ball.idxmax())
    dm = M.loc[f0:f1]

    n_pr = dm[dm.is_presser].groupby(level=0).size().reindex(span.index, fill_value=0)
    nearest = np.hypot(dm.pitch_x - dm.ball_pitch_x, dm.pitch_y - dm.ball_pitch_y).groupby(level=0).min()
    main_id = dm.groupby("track_id").p_ball.mean().idxmax()
    main_name = name_of["name"].get(main_id, f"track {main_id}")

    bx0 = span.ball_pitch_x.iloc[0]
    x_press = bx0 if att_dir[pt] == 1 else PITCH_L - bx0           # 0 = presser's own goal line
    height = "low" if x_press < PITCH_L / 3 else "mid" if x_press < 2 * PITCH_L / 3 else "high"

    s = poss_series.loc[:f0 - 1]
    mine = s[(s == pt).fillna(False).astype(bool)]
    since = (f0 - mine.index.max()) / FPS if len(mine) else np.nan
    trigger = "counterpress" if since <= CTR_WINDOW_S else "set_press"

    w = bm.loc[f1 + 1: f1 + int(OUT_WINDOW_S * FPS)]
    if len(w) == 0:
        outcome, t_reg = "unknown", np.nan
    else:
        regain = ((w.possession_team == pt).fillna(False).astype(bool) & ~w.is_stoppage.astype(bool)).to_numpy()
        stop = w.is_stoppage.to_numpy(bool)
        i_reg = np.flatnonzero(regain); i_stp = np.flatnonzero(stop)
        if len(i_reg) and (not len(i_stp) or i_reg[0] <= i_stp[0]):
            outcome, t_reg = "regain", (w.index[i_reg[0]] - f0) / FPS
        elif len(i_stp):
            outcome, t_reg = "stoppage_out", np.nan
        else:
            outcome, t_reg = "escaped", np.nan

    # shape of the press at the peak frame (only visible outfield players)
    Pf = P.loc[fpk:fpk]
    xy = Pf[(Pf.team == pt) & (Pf.role == "player")][["pitch_x", "pitch_y"]].to_numpy()
    depth = width = hull = line_h = near10 = near15 = mean_d = np.nan
    if len(xy) >= 4:
        bxp, byp = span.loc[fpk, "ball_pitch_x"], span.loc[fpk, "ball_pitch_y"]
        depth, width = np.ptp(xy[:, 0]), np.ptp(xy[:, 1])
        try: hull = ConvexHull(xy).volume
        except Exception: pass
        line_h = (xy[:, 0] if att_dir[pt] == 1 else PITCH_L - xy[:, 0]).mean()
        dd = np.hypot(xy[:, 0] - bxp, xy[:, 1] - byp)
        near10, near15, mean_d = int((dd < 10).sum()), int((dd < 15).sum()), dd.mean()

    rows.append(dict(event_id=eid, start=f0, end=f1, peak_frame=fpk, press_team=pt, attacked_team=at,
                     dur_s=e.dur_s, p_peak=span.p_ball.max(), p_mean=span.p_ball.mean(), p_carrier=span.p_carrier.mean(),
                     n_pressers_mean=n_pr.mean(), n_pressers_max=n_pr.max(), nearest_m=nearest.mean(),
                     main_presser=main_name, x_press=x_press, height=height, trigger=trigger, since_loss_s=since,
                     outcome=outcome, t_regain_s=t_reg, depth=depth, width=width, hull_area=hull,
                     line_height=line_h, near10=near10, near15=near15, mean_dist_ball=mean_d))
ev_stats = pd.DataFrame(rows)
ev_stats["team_name"] = ev_stats.press_team.map(TEAM)
print(ev_stats.outcome.value_counts().to_string())
ev_stats.sort_values("p_peak", ascending=False).head(10)[
    ["event_id", "team_name", "start", "dur_s", "p_peak", "n_pressers_max", "main_presser", "trigger", "outcome"]]

## 6. Per-team averages

In [ ]:
poss_min = fr.possession_team.value_counts() / FPS / 60          # minutes each team HAS the ball (usable frames)
rows = []
for t, name in TEAM.items():
    d = ev_stats[ev_stats.press_team == t]
    if d.empty: continue
    rows.append({
        "n_events": len(d),
        "events_per_10min_opp_poss": len(d) / poss_min.get(1 - t, np.nan) * 10,
        "dur_s": d.dur_s.mean(), "p_peak": d.p_peak.mean(), "p_mean": d.p_mean.mean(),
        "pressers_mean": d.n_pressers_mean.mean(), "nearest_m": d.nearest_m.mean(),
        "press_height_x (m from own goal)": d.x_press.mean(),
        "share_high": (d.height == "high").mean(), "share_mid": (d.height == "mid").mean(), "share_low": (d.height == "low").mean(),
        "counterpress_share": (d.trigger == "counterpress").mean(),
        "regain_rate_3s": (d.outcome == "regain").mean(), "time_to_regain_s": d.t_regain_s.mean(),
        "shape_depth_m": d.depth.mean(), "shape_width_m": d.width.mean(), "hull_area_m2": d.hull_area.mean(),
        "line_height_x": d.line_height.mean(), "players_within_10m": d.near10.mean(), "mean_dist_to_ball": d.mean_dist_ball.mean(),
    })
    rows[-1]["team"] = name
summ = pd.DataFrame(rows).set_index("team")
display(summ.T.round(2))

print("\npress height x outcome:")
display(pd.crosstab([ev_stats.team_name, ev_stats.height], ev_stats.outcome))

ev_stats.to_csv(OUT_DIR / "pressure_events.csv", index=False)
summ.to_csv(OUT_DIR / "team_summary.csv")
print("saved to", OUT_DIR)

### Average shape of the press
Positions of the pressing team's outfield players relative to the ball at the peak frame (all events pooled, rotated so the pressing team attacks to the right).

In [ ]:
COL = {0: "#d62728", 1: "#1f4e9c"}
fig, axes = plt.subplots(1, len(TEAM), figsize=(6 * len(TEAM), 5), squeeze=False)
for ax, (t, name) in zip(axes[0], TEAM.items()):
    xs, ys = [], []
    for e in ev_stats[ev_stats.press_team == t].itertuples():
        Pf = P.loc[e.peak_frame:e.peak_frame]
        Pf = Pf[(Pf.team == t) & (Pf.role == "player")]
        bx, by = fr.loc[e.peak_frame, ["ball_pitch_x", "ball_pitch_y"]]
        sgn = att_dir[t]
        xs += list(((Pf.pitch_x - bx) * sgn).to_numpy()); ys += list(((Pf.pitch_y - by) * sgn).to_numpy())
    if xs:
        hb = ax.hexbin(xs, ys, gridsize=22, extent=(-40, 40, -30, 30), cmap="viridis", mincnt=1)
        plt.colorbar(hb, ax=ax, label="player-frames")
    ax.scatter([0], [0], c="white", edgecolors="black", s=90, zorder=5)
    ax.annotate("", xy=(20, -27), xytext=(-5, -27), arrowprops=dict(arrowstyle="->", lw=2))
    ax.set_xlim(-40, 40); ax.set_ylim(30, -30); ax.set_aspect("equal")
    ax.set_title(f"{name} pressing shape (ball = white dot; {name} attacks to the right)")
plt.tight_layout(); plt.savefig(OUT_DIR / "press_shape_avg.png", dpi=150); plt.show()

## 7. 2D animation of one event
Press team's convex hull (dashed), pressers circled in yellow with lines to the ball (thicker = more pressure), and the pressure oval around the ball.

In [ ]:
NAME = TEAM

def draw_pitch(ax):
    ax.set_facecolor("#2e7d32")
    kw = dict(fill=False, ec="white", lw=1.2, zorder=1)
    ax.add_patch(Rectangle((0, 0), PITCH_L, PITCH_W, **kw)); ax.plot([PITCH_L / 2] * 2, [0, PITCH_W], c="white", lw=1.2, zorder=1)
    ax.add_patch(Circle((PITCH_L / 2, PITCH_W / 2), 9.15, **kw))
    for x0, x1 in [(0, 16.5), (PITCH_L - 16.5, PITCH_L)]: ax.add_patch(Rectangle((x0, 13.84), x1 - x0, 40.32, **kw))
    for x0, x1 in [(0, 5.5), (PITCH_L - 5.5, PITCH_L)]: ax.add_patch(Rectangle((x0, 24.84), x1 - x0, 18.32, **kw))
    ax.set_xlim((PITCH_L + 2, -2) if X_FLIP else (-2, PITCH_L + 2))
    ax.set_ylim((PITCH_W + 2, -2) if Y_DOWN else (-2, PITCH_W + 2))
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])

def pressure_oval(cx, cy, gx, gy, d_front=D_FRONT, d_back=D_BACK, n=90):
    th = np.linspace(0, 2 * np.pi, n); z = (1 + np.cos(th)) / 2
    L = d_back + (d_front - d_back) * (z ** 3 + 0.3 * z) / 1.3
    base = np.arctan2(gy - cy, gx - cx)
    return np.c_[cx + L * np.cos(base + th), cy + L * np.sin(base + th)]

def animate_event(eid, pad_s=PAD_S, step=2, out=None):
    e = ev_stats.loc[ev_stats.event_id == eid].iloc[0]
    pt, at = int(e.press_team), int(e.attacked_team)
    f_lo, f_hi = max(0, int(e.start - pad_s * FPS)), int(e.end + pad_s * FPS)
    frames = list(range(f_lo, f_hi + 1, step))
    ballxy = bm[["ball_pitch_x", "ball_pitch_y"]].reindex(range(f_lo, f_hi + 1)).ffill(limit=5)
    gx = PITCH_L if att_dir[at] == 1 else 0.0
    fig, ax = plt.subplots(figsize=(11, 7.2))

    def update(i):
        f = frames[i]; ax.clear(); draw_pitch(ax)
        Pf = P.loc[f:f]; dmf = M.loc[f:f]
        dm = dict(zip(dmf.track_id, dmf.p_ball)); dteam = dict(zip(dmf.track_id, dmf.team))
        bxy = ballxy.loc[f].to_numpy(float); has_ball = not np.isnan(bxy).any()
        pts = Pf[(Pf.team == pt) & (Pf.role == "player")][["pitch_x", "pitch_y"]].to_numpy()
        if len(pts) >= 3:
            try:
                h = ConvexHull(pts)
                ax.add_patch(Polygon(pts[h.vertices], closed=True, fc=COL[pt], alpha=0.15, ec=COL[pt], lw=1.5, ls="--", zorder=1.5))
            except Exception: pass
        if has_ball:
            ax.add_patch(Polygon(pressure_oval(bxy[0], bxy[1], gx, PITCH_W / 2), closed=True,
                                 fc="yellow", alpha=0.18, ec="yellow", lw=1, zorder=1.6))
        for t in TEAM:
            sub = Pf[Pf.team == t]
            ax.scatter(sub.pitch_x, sub.pitch_y, s=170, c=COL[t], edgecolors="white", zorder=3)
            for r in sub.itertuples():
                if not pd.isna(r.number):
                    ax.text(r.pitch_x, r.pitch_y, str(int(r.number)), color="white", fontsize=7, ha="center", va="center", zorder=4)
        n_pr = 0
        if has_ball:
            for r in Pf[Pf.team == pt].itertuples():
                p = dm.get(r.track_id, 0)
                if p >= PRESSER_MIN and dteam.get(r.track_id) == pt:
                    n_pr += 1
                    ax.plot([r.pitch_x, bxy[0]], [r.pitch_y, bxy[1]], c="yellow", lw=1 + p / 15, alpha=.9, zorder=2)
                    ax.scatter([r.pitch_x], [r.pitch_y], s=380, facecolors="none", edgecolors="yellow", linewidths=2, zorder=5)
            ax.scatter([bxy[0]], [bxy[1]], s=70, c="white", edgecolors="black", zorder=6)
        pb = fr.p_ball.get(f, np.nan)
        phase = "PRESS" if e.start <= f <= e.end else ("before" if f < e.start else "after")
        ax.set_title(f"Event {eid}: {NAME[pt]} press on {NAME[at]} | t={(f - e.start) / FPS:+.1f}s | "
                     f"pressure on ball {0 if np.isnan(pb) else pb:.0f} | pressers {n_pr} | {phase}",
                     fontsize=10, color="red" if phase == "PRESS" else "black")
        ax.annotate("", xy=(gx, PITCH_W - 2), xytext=(PITCH_L / 2, PITCH_W - 2), arrowprops=dict(arrowstyle="->", color="white", lw=2))

    ani = animation.FuncAnimation(fig, update, frames=len(frames), interval=1000 * step / FPS)
    has_ffmpeg = animation.writers.is_available("ffmpeg")
    ext = "mp4" if has_ffmpeg else "gif"
    out = Path(out or OUT_DIR / "animations" / f"event_{eid:03d}_{NAME[pt]}_press.{ext}")
    ani.save(str(out), writer="ffmpeg" if has_ffmpeg else "pillow", fps=FPS / step)
    plt.close(fig); print("saved", out); return out

best = int(ev_stats.sort_values("p_peak", ascending=False).event_id.iloc[0])
animate_event(best)

## 8. Annotated clips from the real video

**First check the alignment** with `preview_frame` (a frame at the peak of the strongest event). The boxes must sit on the right players.
If they are shifted in time, change `VIDEO_FRAME_OFFSET`. If they are scaled or offset in space, the video is not the resolution the pipeline processed.

In [ ]:
BGR = {0: (40, 39, 214), 1: (156, 78, 31)}            # per team id, BGR (red, blue). Edit if you want other colours.
ascii_ = lambda s: unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
PA = players.set_index("frame_idx").sort_index()      # all players/referees with pixel boxes

MM_W, MM_H = 336, 220                                 # minimap size in px

def _mm_pt(x, y, x0, y0):
    px = (PITCH_L - x) if X_FLIP else x
    py = y if Y_DOWN else (PITCH_W - y)
    return int(x0 + 6 + px / PITCH_L * (MM_W - 12)), int(y0 + 6 + py / PITCH_W * (MM_H - 12))

def draw_minimap(img, f, e):
    H = img.shape[0]; x0, y0 = 12, H - MM_H - 12
    roi = img[y0:y0 + MM_H, x0:x0 + MM_W]
    roi[:] = (roi * 0.30 + np.array((46, 125, 50)) * 0.70).astype(np.uint8)
    pts = lambda x, y: _mm_pt(x, y, x0, y0)
    cv2.rectangle(img, pts(0, 0), pts(PITCH_L, PITCH_W), (255, 255, 255), 1)
    cv2.line(img, pts(PITCH_L / 2, 0), pts(PITCH_L / 2, PITCH_W), (255, 255, 255), 1)
    cv2.circle(img, pts(PITCH_L / 2, PITCH_W / 2), int(9.15 / PITCH_L * MM_W), (255, 255, 255), 1)
    for xa, xb in [(0, 16.5), (PITCH_L - 16.5, PITCH_L)]:
        cv2.rectangle(img, pts(xa, 13.84), pts(xb, 54.16), (255, 255, 255), 1)
    Pf = P.loc[f:f]
    dmf = M.loc[f:f]
    pt = None if e is None else int(e.press_team)
    prs = set(dmf[(dmf.team == pt) & dmf.is_presser].track_id) if pt is not None else set()
    bxy = fr.loc[f, ["ball_pitch_x", "ball_pitch_y"]].to_numpy(float) if f in fr.index else None
    if pt is not None:
        xy = Pf[(Pf.team == pt) & (Pf.role == "player")][["pitch_x", "pitch_y"]].to_numpy()
        if len(xy) >= 3:
            try:
                h = ConvexHull(xy)
                cv2.polylines(img, [np.array([pts(*xy[i]) for i in h.vertices], np.int32)], True, (255, 255, 255), 1, cv2.LINE_AA)
            except Exception: pass
    for r in Pf.itertuples():
        p = pts(r.pitch_x, r.pitch_y)
        cv2.circle(img, p, 5, BGR[int(r.team)], -1)
        if r.track_id in prs:
            cv2.circle(img, p, 8, (0, 255, 255), 2)
            if bxy is not None and not np.isnan(bxy).any():
                cv2.line(img, p, pts(*bxy), (0, 255, 255), 1, cv2.LINE_AA)
    if bxy is not None and not np.isnan(bxy).any():
        cv2.circle(img, pts(*bxy), 4, (255, 255, 255), -1); cv2.circle(img, pts(*bxy), 4, (0, 0, 0), 1)
    return img

def annotate_frame(img, f, e=None):
    H, W = img.shape[:2]; s = max(0.55, W / 1920)
    pt = None if e is None else int(e.press_team)
    dmf = M.loc[f:f]
    dm = dict(zip(dmf.track_id, dmf.p_ball)); dteam = dict(zip(dmf.track_id, dmf.team))
    brow = bm.loc[f] if f in bm.index else None
    bpx = None
    if brow is not None and not np.isnan(brow.ball_px_x):
        bpx = (int(brow.ball_px_x), int(brow.ball_px_y))
    n_pr = 0
    for r in PA.loc[f:f].itertuples():
        x1, y1, x2, y2 = int(r.x1), int(r.y1), int(r.x2), int(r.y2)
        if pd.isna(r.team):
            cv2.rectangle(img, (x1, y1), (x2, y2), (170, 170, 170), 1); continue
        t = int(r.team); p = dm.get(r.track_id, 0.0)
        is_pr = pt is not None and t == pt and dteam.get(r.track_id) == pt and p >= PRESSER_MIN
        if is_pr:
            n_pr += 1
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 255), max(2, int(3 * s)))
            num = "" if pd.isna(r.number) else f"#{int(r.number)} "
            last = ascii_(r.name).split()[-1] if isinstance(r.name, str) and r.name else ""
            cv2.putText(img, f"{num}{last} {p:.0f}", (x1, max(14, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6 * s, (0, 255, 255), max(1, int(2 * s)), cv2.LINE_AA)
            if bpx: cv2.line(img, (int(r.foot_px_x), int(r.foot_px_y)), bpx, (0, 255, 255), max(1, int(p / 15)), cv2.LINE_AA)
        else:
            cv2.rectangle(img, (x1, y1), (x2, y2), BGR[t], max(1, int(2 * s)))
    if bpx:
        cv2.circle(img, bpx, int(10 * s) + 2, (0, 0, 0), 3); cv2.circle(img, bpx, int(10 * s), (255, 255, 255), -1)

    pb = fr.p_ball.get(f, np.nan); pbv = 0 if np.isnan(pb) else pb
    inside = e is not None and e.start <= f <= e.end
    cv2.rectangle(img, (0, 0), (W, int(48 * s) + 6), (20, 20, 20), -1)
    head = f"frame {f}" if e is None else (f"Event {int(e.event_id)} | {NAME[pt]} press | t={(f - e.start) / FPS:+.1f}s | "
                                              f"pressers={n_pr} | outcome: {e.outcome}")
    cv2.putText(img, head, (12, int(34 * s) + 2), cv2.FONT_HERSHEY_SIMPLEX, 0.85 * s, (255, 255, 255), max(1, int(2 * s)), cv2.LINE_AA)
    bw = int(320 * s); bx0 = W - bw - 20
    cv2.rectangle(img, (bx0, int(12 * s)), (bx0 + bw, int(36 * s)), (255, 255, 255), 1)
    cv2.rectangle(img, (bx0, int(12 * s)), (bx0 + int(bw * min(pbv, 100) / 100), int(36 * s)), (0, 0, 255) if inside else (0, 200, 0), -1)
    cv2.putText(img, f"pressure {pbv:.0f}", (bx0 - int(170 * s), int(34 * s)), cv2.FONT_HERSHEY_SIMPLEX, 0.7 * s, (255, 255, 255), 1, cv2.LINE_AA)
    return draw_minimap(img, f, e)

def read_frame(f):
    cap = cv2.VideoCapture(str(VIDEO)); cap.set(cv2.CAP_PROP_POS_FRAMES, f + VIDEO_FRAME_OFFSET)
    ok, img = cap.read(); cap.release()
    assert ok, f"cannot read video frame {f + VIDEO_FRAME_OFFSET}"
    return img

def preview_frame(f, eid=None):
    e = None if eid is None else ev_stats.loc[ev_stats.event_id == eid].iloc[0]
    img = annotate_frame(read_frame(f), f, e)
    plt.figure(figsize=(14, 8)); plt.imshow(img[..., ::-1]); plt.axis("off"); plt.show()

best_row = ev_stats.sort_values("p_peak", ascending=False).iloc[0]
preview_frame(int(best_row.peak_frame), int(best_row.event_id))

In [ ]:
def render_clip(eid, out_path=None, pad_s=PAD_S, reencode=True):
    e = ev_stats.loc[ev_stats.event_id == eid].iloc[0]
    pt = int(e.press_team)
    f_lo, f_hi = max(0, int(e.start - pad_s * FPS)), int(e.end + pad_s * FPS)
    out_path = Path(out_path or OUT_DIR / "clips" / f"event_{eid:03d}_{NAME[pt]}_press.mp4")
    tmp = out_path.with_suffix(".tmp.mp4")

    cap = cv2.VideoCapture(str(VIDEO)); assert cap.isOpened(), "cannot open video"
    fps = cap.get(cv2.CAP_PROP_FPS) or FPS
    W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, f_lo + VIDEO_FRAME_OFFSET)
    vw = cv2.VideoWriter(str(tmp), cv2.VideoWriter_fourcc(*"mp4v"), fps * SLOWMO, (W, H))
    for f in range(f_lo, f_hi + 1):
        ok, img = cap.read()
        if not ok: break
        vw.write(annotate_frame(img, f, e))
    cap.release(); vw.release()

    if reencode and shutil.which("ffmpeg"):                       # H.264 plays everywhere (browser, PowerPoint)
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(tmp), "-c:v", "libx264",
                        "-pix_fmt", "yuv420p", "-crf", "20", str(out_path)], check=True)
        tmp.unlink()
    else:
        tmp.replace(out_path)
    print("saved", out_path); return out_path

def export(event_ids, clips=True, anims=True):
    for eid in event_ids:
        eid = int(eid)
        if clips: render_clip(eid)
        if anims: animate_event(eid)

# test on the strongest event first
render_clip(int(best_row.event_id))

## 9. Batch export
Pick the events you want. A few examples are below. Uncomment one and run.

In [ ]:
top = ev_stats.sort_values("p_peak", ascending=False)

# export(top.event_id.head(10))                                                   # 10 strongest presses
# export(ev_stats.query("outcome == 'regain'").sort_values("t_regain_s").event_id.head(10))   # fastest regains
# export(ev_stats.query("trigger == 'counterpress'").event_id.head(10))           # counterpresses
# export(ev_stats.query("team_name == 'BAR' and height == 'high'").event_id.head(10))
# export(ev_stats.event_id, anims=False)                                          # EVERY event as a clip (slow)

print(f"{len(ev_stats)} events available -> pick ids from ev_stats / pressure_events.csv")